# microWakeWord Training Notebook — "hey frank"

Tuned for:
- **Wake word:** `hey frank` (phonetic: `'hey frank'`)
- **Hardware:** Local NVIDIA GPU via WSL2 (tested on 12 GB VRAM)
- **Target:** M5Stack devices via ESPHome

Based on the official `basic_training_notebook.ipynb` with practical improvements.
**Run in WSL2 with the wakeword-env venv. Use Python 3.12.**

## Quick-edit constants (top of each relevant cell, marked `# <-- EDIT`):
| Setting | Value | Notes |
|---|---|---|
| `target_word` | `'hey frank'` | Phonetic — change if pronunciation sounds off |
| `MAX_SAMPLES` | `50_000` | Good for NVIDIA GPU; reduce to 25k if storage is tight |
| `PIPER_BATCH` | `256` | Piper TTS clips per GPU pass — reduce to 128 if CUDA OOM during sample generation |
| `batch_size` (YAML) | `256` | Training mini-batch size — tuned for 12 GB VRAM; reduce to 128 if OOM during training |
| Training steps | `[30000, 20000]` | 50k total; reduce to `[5000]` for a quick test |
| `target_minimization` | `2.0` FA/hr | Lower (e.g. 0.5) for stricter false accept control |

**Upstream warning:** The model may not be usable without experimentation. Treat this as a starting point!

In [ ]:
# ── Install microWakeWord ────────────────────────────────────────────────────
# RESTART the kernel after this cell finishes!
# On re-runs: microWakeWord/ clone is skipped automatically if it already exists.

import platform, os

if platform.system() == "Darwin":
    # macOS / Apple Silicon only — not needed for Colab/NVIDIA
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# audio-metadata fork unpins `attrs` so Jupyter doesn't break
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

if not os.path.exists('microWakeWord'):
    !git clone https://github.com/kahrendt/microWakeWord
else:
    print("microWakeWord already cloned, skipping")

!pip install -e ./microWakeWord

print("\n✅ Done. RESTART YOUR KERNEL before running any other cells!")

In [ ]:
# ── Wake word config + single preview sample ─────────────────────────────────
#
# PHONETICS NOTE: 'hey frank' is usually clear as-is.
#   If the preview sounds off, try: 'hey fraenk', 'hey frahnk'
#
# MAX_SAMPLES: 50k is the sweet spot for GPU training.
#   50_000 = best quality (matches official models)
#   25_000 = good balance if disk space is tight (~5GB)
#   10_000 = quick iteration / testing only
#
# PIPER_BATCH: Piper TTS clips generated per GPU pass.
#   256 = good for 8GB+ VRAM (RTX 3070/3080/4070 etc.)
#   128 = safer for 6–8GB VRAM

# RUN IN GOOGLE COLAB

target_word  = 'hey frank'  # <-- EDIT if pronunciation sounds wrong
MAX_SAMPLES  = 50_000       # <-- EDIT: 50k recommended for GPU
PIPER_BATCH  = 256          # <-- EDIT: reduce to 128 if CUDA out-of-memory

import os, sys, platform, urllib.request
from IPython.display import Audio

MODEL_PATH = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'

# ── Clone repo ────────────────────────────────────────────────────────────────
if not os.path.exists('./piper-sample-generator'):
    if platform.system() == 'Darwin':
        !git clone -b mps-support https://github.com/kahrendt/piper-sample-generator
    else:
        !git clone https://github.com/rhasspy/piper-sample-generator

# ── Download model ────────────────────────────────────────────────────────────
model_url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
if not os.path.exists(MODEL_PATH):
    print('Downloading TTS model (~300 MB)...')
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    urllib.request.urlretrieve(model_url, MODEL_PATH)
    print('✅ Model downloaded')
else:
    print('✅ Model already present')

# ── Install dependencies (always runs so re-runs after kernel restart work) ──
!pip install torch torchaudio piper-phonemize-cross==1.2.1 piper-tts

if 'piper-sample-generator/' not in sys.path:
    sys.path.append('piper-sample-generator/')

# ── Generate 1 sample to verify pronunciation ─────────────────────────────────
!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--model {MODEL_PATH} \
--max-samples 1 \
--batch-size 1 \
--output-dir generated_samples

print()
print('Listen below — should sound like "hey frank" clearly.')
print('If not, edit target_word above and re-run before proceeding.')
Audio(filename='generated_samples/0.wav', autoplay=True)

✅ Model already present
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 8.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 61.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 11.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 57.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 43.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 32.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 51.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━

In [ ]:
# ── Generate full training sample set ────────────────────────────────────────
# Generates MAX_SAMPLES TTS clips of 'hey amy' using the libritts multi-speaker
# model (varied voices = more robust model).
#
# At PIPER_BATCH=256 on a Colab T4 GPU this takes roughly:
#   10k samples ~  5 min
#   25k samples ~ 12 min
#   50k samples ~ 25 min
#
# Piper parameters to experiment with later:
#   --noise-scale   (default 0.667)  phoneme-level variation; try 0.5–0.9
#   --noise-scale-w (default 0.8)    rhythm/duration variation; try 0.6–1.0
#   --length-scale  (default 1.0)    speaking rate; try 0.85–1.15

# RUN IN GOOGLE COLAB

!python3 piper-sample-generator/generate_samples.py "{target_word}" \
--model {MODEL_PATH} \
--max-samples {MAX_SAMPLES} \
--batch-size {PIPER_BATCH} \
--output-dir generated_samples

n = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
print(f'\n✅ {n} samples in ./generated_samples/')

DEBUG:__main__:Loading piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:__main__:Successfully loaded the model
DEBUG:__main__:CUDA available, using GPU
DEBUG:__main__:Batch 1/195 complete
DEBUG:__main__:Batch 2/195 complete
DEBUG:__main__:Batch 3/195 complete
DEBUG:__main__:Batch 4/195 complete
DEBUG:__main__:Batch 5/195 complete
DEBUG:__main__:Batch 6/195 complete
DEBUG:__main__:Batch 7/195 complete
DEBUG:__main__:Batch 8/195 complete
DEBUG:__main__:Batch 9/195 complete
DEBUG:__main__:Batch 10/195 complete
DEBUG:__main__:Batch 11/195 complete
DEBUG:__main__:Batch 12/195 complete


In [13]:
# ── Download augmentation audio ───────────────────────────────────────────────
# Downloads room impulse responses (reverb simulation) and background noise.
# Uses decode=False + soundfile to avoid torchcodec API incompatibilities.
#
# NOTE: Mixed licenses — suitable for non-commercial personal use only.

import datasets, scipy, os, sys, io, urllib.request, zipfile, numpy as np
import librosa, soundfile as sf, fsspec
from pathlib import Path
from tqdm import tqdm

def _decode_audio(audio_info, target_sr=16000):
    """Decode audio from a datasets Audio(decode=False) row using soundfile.
    Handles both inline bytes and hf:// path references."""
    data = audio_info.get('bytes')
    if not data:
        path = audio_info.get('path') or ''
        if path.startswith('hf://'):
            with fsspec.open(path, 'rb') as f:
                data = f.read()
        else:
            data = open(path, 'rb').read()
    arr, sr = sf.read(io.BytesIO(data), dtype='float32', always_2d=False)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)  # stereo -> mono
    if sr != target_sr:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
    return arr

# ── MIT Room Impulse Responses ───────────────────────────────────────────────
rir_wavs = list(Path('mit_rirs').glob('*.wav')) if Path('mit_rirs').exists() else []
if len(rir_wavs) < 10:
    print(f'Downloading MIT RIRs (found {len(rir_wavs)} existing)...')
    os.makedirs('./mit_rirs', exist_ok=True)
    rir_ds = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses', split='train', streaming=True
    )
    rir_ds = rir_ds.cast_column('audio', datasets.Audio(decode=False))
    for i, row in enumerate(tqdm(rir_ds)):
        # path may be hf:// URL or None — use index as fallback filename
        raw_name = (row['audio'].get('path') or '').split('/')[-1].split('\\')[-1]
        name = raw_name if raw_name.lower().endswith('.wav') else f'rir_{i:04d}.wav'
        try:
            arr = _decode_audio(row['audio'])
            scipy.io.wavfile.write(f"mit_rirs/{name}", 16000, (arr * 32767).astype(np.int16))
        except Exception as e:
            print(f'  ⚠️  Skipped row {i}: {e}')
    n = len(list(Path('mit_rirs').glob('*.wav')))
    print(f'✅ MIT RIRs done ({n} files)')
else:
    print(f'✅ MIT RIRs already present ({len(rir_wavs)} files)')

# ── AudioSet (~2000 clips, streaming — old tar files removed Sep 2025) ────────
AUDIOSET_CLIPS = 2000
if not os.path.exists('audioset_16k') or len(list(Path('audioset_16k').glob('*.wav'))) < 100:
    print(f'Downloading AudioSet ({AUDIOSET_CLIPS} clips)...')
    os.makedirs('audioset_16k', exist_ok=True)
    ds = datasets.load_dataset(
        'agkphysics/AudioSet', 'balanced', split='train', streaming=True, trust_remote_code=True
    )
    ds = ds.cast_column('audio', datasets.Audio(decode=False))
    for i, row in enumerate(tqdm(ds, total=AUDIOSET_CLIPS)):
        if i >= AUDIOSET_CLIPS:
            break
        try:
            arr = _decode_audio(row['audio'])
            scipy.io.wavfile.write(f"audioset_16k/{i:05d}.wav", 16000, (arr * 32767).astype(np.int16))
        except Exception:
            pass
    print('✅ AudioSet done')
else:
    print('✅ AudioSet already present')

# ── Free Music Archive (xsmall) ───────────────────────────────────────────────
if not os.path.exists('fma'):
    print('Downloading FMA xsmall...')
    os.makedirs('fma', exist_ok=True)
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip',
        'fma/fma_xs.zip'
    )
    with zipfile.ZipFile('fma/fma_xs.zip') as zf:
        zf.extractall('fma')

if not os.path.exists('fma_16k') or len(list(Path('fma_16k').glob('*.wav'))) < 100:
    print('Converting FMA to 16kHz WAV...')
    os.makedirs('fma_16k', exist_ok=True)
    for p in tqdm(sorted(Path('fma').glob('**/*.mp3'))):
        try:
            arr, _ = librosa.load(str(p), sr=16000, mono=True)
            scipy.io.wavfile.write(f"fma_16k/{p.stem}.wav", 16000, (arr * 32767).astype(np.int16))
        except Exception:
            pass
    print('✅ FMA done')
else:
    print('✅ FMA already present')

270it [00:20, 13.22it/s]

✅ MIT RIRs done (270 files)
✅ AudioSet already present
✅ FMA already present


In [1]:
# ── Configure augmentation pipeline ──────────────────────────────────────────
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from pathlib import Path
import os

# ── Verify data directories ───────────────────────────────────────────────────
print(f'Working directory: {os.getcwd()}')
for d in ['mit_rirs', 'fma_16k', 'audioset_16k', 'generated_samples']:
    p = Path(d)
    if p.exists():
        n = len(list(p.glob('*.wav')))
        print(f'  ✅ {d}: {n} .wav files  ({p.resolve()})')
    else:
        print(f'  ❌ {d}: NOT FOUND  (expected at {p.resolve()})')

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15,
        'TanhDistortion':        0.10,
        'PitchShift':            0.15,
        'BandStopFilter':        0.10,
        'AddColorNoise':         0.20,
        'AddBackgroundNoise':    0.85,
        'Gain':                  1.00,
        'GainTransition':        0.25,
        'RIR':                   0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['fma_16k', 'audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=20,
    min_jitter_s=0.10,
    max_jitter_s=0.50,
)

print('✅ Augmentation pipeline ready')

I0000 00:00:1773016348.188063     451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773016348.681015     451 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773016350.125935     451 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Working directory: /mnt/c/Users/khmal/inventr.io/micro-wake-word/notebooks
  ✅ mit_rirs: 270 .wav files  (/mnt/c/Users/khmal/inventr.io/micro-wake-word/notebooks/mit_rirs)
  ✅ fma_16k: 210 .wav files  (/mnt/c/Users/khmal/inventr.io/micro-wake-word/notebooks/fma_16k)
  ✅ audioset_16k: 2000 .wav files  (/mnt/c/Users/khmal/inventr.io/micro-wake-word/notebooks/audioset_16k)
  ✅ generated_samples: 50000 .wav files  (/mnt/c/Users/khmal/inventr.io/micro-wake-word/notebooks/generated_samples)
✅ Augmentation pipeline ready


In [2]:
# ── Preview an augmented sample ───────────────────────────────────────────────
# Run this a few times. You should still be able to hear 'hey amy' even through
# heavy noise/reverb — if it's completely unintelligible, reduce the SNR range
# (raise background_min_snr_db toward 0) or lower the RIR probability.

from IPython.display import Audio
from microwakeword.audio.audio_utils import save_clip

random_clip    = clips.get_random_clip()
augmented_clip = augmenter.augment_clip(random_clip)
save_clip(augmented_clip, 'augmented_clip.wav')

print('Augmented preview (re-run for variety):')
Audio(filename='augmented_clip.wav', autoplay=True)

Augmented preview (re-run for variety):


In [3]:
# ── Generate spectrogram features ─────────────────────────────────────────────
# Converts augmented clips into 40-band spectrogram features (same format the
# M5Stack micro_speech preprocessor produces at runtime).
#
# training:   slide_frames=10, repeat=3 — each clip generates 3 augmented versions
# validation: slide_frames=10, repeat=1
# testing:    slide_frames=1,  repeat=1 — simulates actual streaming inference
#
# At 50k samples on an NVIDIA GPU this takes roughly 20–40 minutes.
# Already-generated splits are skipped automatically on re-runs.

import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

os.makedirs('generated_augmented_features', exist_ok=True)

split_config = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in split_config.items():
    out_dir   = f'generated_augmented_features/{split}'
    mmap_path = f'{out_dir}/wakeword_mmap'

    # Check for valid (non-empty) existing mmap
    if os.path.exists(mmap_path):
        mmap_files = list(os.scandir(mmap_path))
        if mmap_files:
            print(f'⏭  {split}: already exists ({len(mmap_files)} files), skipping')
            continue
        else:
            print(f'⚠️  {split}: mmap dir exists but is empty — removing and regenerating')
            shutil.rmtree(mmap_path)

    os.makedirs(out_dir, exist_ok=True)
    print(f'\n⚙️  Generating {split} (slide_frames={cfg["slide_frames"]}, repeat={cfg["repetition"]})...')

    try:
        spectrograms = SpectrogramGeneration(
            clips=clips,
            augmenter=augmenter,
            slide_frames=cfg['slide_frames'],
            step_ms=10,
        )
        RaggedMmap.from_generator(
            out_dir=mmap_path,
            sample_generator=spectrograms.spectrogram_generator(
                split=cfg['split_name'],
                repeat=cfg['repetition'],
            ),
            batch_size=200,
            verbose=True,
        )
        print(f'✅ {split} done')
    except Exception:
        print(f'\n❌ {split} FAILED — full traceback:')
        traceback.print_exc()
        # Remove partial mmap so next run retries cleanly
        if os.path.exists(mmap_path):
            shutil.rmtree(mmap_path)
            print(f'   Removed partial mmap at {mmap_path}')
        break

print('\n✅ All feature sets ready')


⚙️  Generating training (slide_frames=10, repeat=3)...


0it [00:00, ?it/s]

✅ training done

⚙️  Generating validation (slide_frames=10, repeat=1)...


0it [00:00, ?it/s]

✅ validation done

⚙️  Generating testing (slide_frames=1, repeat=1)...


0it [00:00, ?it/s]

✅ testing done

✅ All feature sets ready


In [ ]:
# ── Download pre-generated negative datasets ──────────────────────────────────
# Real-world ambient/speech recordings pre-processed by the microWakeWord project.
#
#  speech            — general speech (hardest negatives for 'hey amy')
#  dinner_party      — multi-speaker conversation + background noise
#  no_speech         — ambient/music with no speech
#  dinner_party_eval — held-out eval set for ambient false-positives-per-hour metric

import os, urllib.request, zipfile

if not os.path.exists('./negative_datasets'):
    print('Downloading negative datasets...')
    os.makedirs('./negative_datasets', exist_ok=True)
    link_root = 'https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/'
    for fname in ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']:
        zip_path = f'negative_datasets/{fname}'
        urllib.request.urlretrieve(link_root + fname, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall('negative_datasets')
        print(f'  ✅ {fname}')
    print('✅ All negative datasets ready')
else:
    print('✅ Negative datasets already present')

In [4]:
# ── Write training configuration YAML ────────────────────────────────────────
# Tuned for 'hey frank' on a local NVIDIA GPU (12 GB VRAM).
#
# Key parameters to experiment with:
#
#  training_steps: [30000, 20000] = 50k total across 2 phases.
#    Phase 1 (LR=0.001):  fast convergence
#    Phase 2 (LR=0.0001): fine-tuning / squeezing out false accepts
#    For a quick sanity-check run: change to [5000]
#    For aggressive false-accept reduction: try [25000, 15000, 10000] (3 phases)
#
#  negative_class_weight: 25 / 30
#    This is the most impactful knob for false accept rate.
#    Increase toward 40–50 if the model still triggers on normal speech.
#    Decrease toward 10–15 if it barely triggers on the real wake word.
#
#  target_minimization: 2.0 FA/hr
#    The trainer first tries to get ambient false accepts under this threshold,
#    THEN maximizes recall. Set lower (e.g. 0.5) for a quieter device.
#
#  batch_size: 256
#    Tuned for 12 GB VRAM. Reduce to 128 if you hit OOM during training.

import yaml, os

config = {}
config['window_step_ms'] = 10
config['train_dir']      = 'trained_models/hey_frank'

config['features'] = [
    {
        'features_dir':         'generated_augmented_features',
        'sampling_weight':      3.0,
        'penalty_weight':       2.0,
        'truth':                True,
        'truncation_strategy':  'truncate_start',
        'type':                 'mmap',
    },
    {
        'features_dir':         'negative_datasets/speech',
        'sampling_weight':      10.0,
        'penalty_weight':       2.0,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        'features_dir':         'negative_datasets/dinner_party',
        'sampling_weight':      10.0,
        'penalty_weight':       1.5,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        'features_dir':         'negative_datasets/no_speech',
        'sampling_weight':      5.0,
        'penalty_weight':       1.0,
        'truth':                False,
        'truncation_strategy':  'random',
        'type':                 'mmap',
    },
    {
        # Eval-only — sampling_weight=0 means never in training batches
        'features_dir':         'negative_datasets/dinner_party_eval',
        'sampling_weight':      0.0,
        'penalty_weight':       1.0,
        'truth':                False,
        'truncation_strategy':  'split',
        'type':                 'mmap',
    },
]

# Two-phase training schedule (50k total — recommended for good model quality)
config['training_steps']        = [30000, 20000]  # <-- EDIT: [5000] for quick test
config['positive_class_weight'] = [1,     1    ]
config['negative_class_weight'] = [25,    30   ]  # <-- EDIT: increase if too many false accepts
config['learning_rates']        = [0.001, 0.0001]
config['batch_size']            = 256  # <-- EDIT: reduce to 128 if OOM during training

# SpecAugment — improves generalization
config['time_mask_max_size'] = [5, 5]
config['time_mask_count']    = [1, 1]
config['freq_mask_max_size'] = [3, 3]
config['freq_mask_count']    = [1, 1]

config['eval_step_interval'] = 500
config['clip_duration_ms']   = 1500

# Best-weights selection: minimize FA/hr first, then maximize recall
config['target_minimization'] = 2.0   # <-- EDIT: acceptable false accepts per hour
config['minimization_metric'] = 'ambient_false_positives_per_hour'
config['maximization_metric'] = 'average_viable_recall'

os.makedirs('trained_models/hey_frank', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)

print('✅ training_parameters.yaml written')
print(f'   Phases: {len(config["training_steps"])}  |  Total steps: {sum(config["training_steps"])}  |  Batch: {config["batch_size"]}')

✅ training_parameters.yaml written
   Phases: 2  |  Total steps: 50000  |  Batch: 256


In [ ]:
# ── Train the model ───────────────────────────────────────────────────────────
# NOTE: Run this cell via the WSL CLI for long training runs — it's more stable
# than keeping Jupyter open for hours. See notebook header for instructions.
#
# Use --train 0 to skip training and only export/evaluate a saved checkpoint.

# use tmux new -s training
# or tmux attach -t training
# then: 

'''
export XLA_FLAGS='--xla_gpu_autotune_level=0'
export PATH="/home/malonestar/wakeword-env/lib/python3.12/site-packages/nvidia/cuda_nvcc/bin:$PATH"

nohup python3 -m microwakeword.model_train_eval \
  --training_config "training_parameters.yaml" \
  --train 1 \
  --restore_checkpoint 0 \
  --test_tf_nonstreaming 0 \
  --test_tflite_nonstreaming 0 \
  --test_tflite_nonstreaming_quantized 0 \
  --test_tflite_streaming 0 \
  --test_tflite_streaming_quantized 1 \
  --use_weights "best_weights" \
  mixednet \
  --pointwise_filters "64,64,64,64" \
  --repeat_in_block "1, 1, 1, 1" \
  --mixconv_kernel_sizes "[5], [7,11], [9,15], [23]" \
  --residual_connection "0,0,0,0" \
  --first_conv_filters 32 \
  --first_conv_kernel_size 5 \
  --stride 3 \
  > training.log 2>&1 &

tail -f training.log

'''

import sys, os

# Use pip-bundled ptxas 12.9 instead of system ptxas 12.4
# (required for RTX 5000 series / Blackwell CC 12.0)
_ptxas_dir = os.path.join(os.path.dirname(sys.executable),
             '../lib/python3.12/site-packages/nvidia/cuda_nvcc/bin')
_ptxas_dir = os.path.normpath(_ptxas_dir)
os.environ['PATH'] = _ptxas_dir + ':' + os.environ.get('PATH', '')
os.environ['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

# Ensure TensorBoard is present (required by tf.summary)
!"{sys.executable}" -m pip install -q tensorboard

!"{sys.executable}" -m microwakeword.model_train_eval \
--training_config "training_parameters.yaml" \
--train 1 \
--restore_checkpoint 1 \
--test_tf_nonstreaming 0 \
--test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 \
--test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 \
--use_weights "best_weights" \
mixednet \
--pointwise_filters "64,64,64,64" \
--repeat_in_block  "1, 1, 1, 1" \
--mixconv_kernel_sizes "[5], [7,11], [9,15], [23]" \
--residual_connection "0,0,0,0" \
--first_conv_filters 32 \
--first_conv_kernel_size 5 \
--stride 3

print('\n✅ Training complete!')

In [ ]:
# ── Locate the trained model + print ESPHome manifest template ───────────────
# The .tflite is your deployable model.
# You need a companion JSON manifest to use it in ESPHome / M5Stack.
# A starter template is printed below — fill in the probability_cutoff
# based on the test metrics printed by the training cell above.
#
# See full docs: https://esphome.io/components/micro_wake_word
# See examples:  https://github.com/esphome/micro-wake-word-models/tree/main/models/v2

import os, json

tflite_path = 'trained_models/hey_frank/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'

if os.path.exists(tflite_path):
    size_kb = os.path.getsize(tflite_path) / 1024
    print(f'✅ Model: {tflite_path}')
    print(f'   Size:  {size_kb:.1f} KB')

    manifest = {
        'type': 'micro_wake_word_model',
        'wake_word': 'hey frank',
        'author': 'your-name',
        'website': '',
        'model': 'hey_frank.tflite',
        'version': 1,
        'micro': {
            # Adjust based on test metrics — lower = more sensitive / more false accepts
            'probability_cutoff':    0.5,   # <-- EDIT based on training output
            'sliding_window_average': 10,
            'tensor_arena_size':     30000,
        }
    }

    print('\n── ESPHome manifest template (hey_frank.json) ──')
    print(json.dumps(manifest, indent=2))

    with open('hey_frank.json', 'w') as f:
        json.dump(manifest, f, indent=2)
    print('\n✅ hey_frank.json written to working directory')
    print(f'   Copy hey_frank.tflite and hey_frank.json to your ESPHome config directory.')
else:
    print('❌ Model not found — check training output above for errors.')